# MSE 446 Project - Team 2
## GMM Song Recommendation Model
Loads cleaned data from  (produced by ).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load cleaned dataset produced by MSE446_Group2_DataCleaning.ipynb
songs = pd.read_csv("datasets/songs_cleaned.csv")
print(f"Loaded {len(songs)} songs with {songs.shape[1]} features")
songs.head()

Loaded 4493 songs with 150 features


,track_id,energy,tempo,danceability,loudness,liveness,valence,track_artist,speechiness,track_popularity,...,playlist_genre_pop,playlist_genre_punk,playlist_genre_r&b,playlist_genre_reggae,playlist_genre_rock,playlist_genre_soca,playlist_genre_soul,playlist_genre_turkish,playlist_genre_wellness,playlist_genre_world
0,00Coyxt9mTec1acC52qtWa,0.736420,0.329870,0.634822,0.898334,0.083507,0.502,TAEIL,0.009170,0.494382,...,0,0,0,0,0,0,0,0,0,0
1,00DPAwQ3NkWs6PZKNxy7Pi,0.899779,0.433735,0.782632,0.880394,0.168058,0.675,Olamilekan Akamo,0.069716,0.213483,...,0,0,0,0,0,0,0,0,0,0
2,00Gbi2ytn6ZmA1ObVcPT93,0.928843,0.454284,0.579394,0.866443,0.670146,0.678,Smith & Thell,0.024307,0.224719,...,1,0,0,0,0,0,0,0,0,0
3,00JOgmWv6RmkgwPxdYScnf,0.472839,0.480056,0.387023,0.748780,0.200418,0.067,Greg Edmonson,0.017015,0.179775,...,0,0,0,0,0,0,0,0,0,0
4,00Mb3DuaIH1kjrwOku9CGU,0.901784,0.526440,0.465276,0.883876,0.351775,0.484,Avril Lavigne,0.029058,0.741573,...,0,1,0,0,0,0,0,0,0,0


In [3]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------
# GMM-BASED SONG RECOMMENDER
# -------------------------------

songs_gmm = songs.copy()

# Audio features used for GMM (excludes popularity — model is blind to fame)
base_features = [
    'energy', 'tempo', 'danceability', 'loudness', 'liveness',
    'valence', 'speechiness', 'acousticness', 'instrumentalness'
]

genre_cols   = [c for c in songs_gmm.columns if c.startswith('playlist_genre_')]
time_sig_cols = [c for c in songs_gmm.columns if c.startswith('time_signature_')]

feature_cols = base_features + genre_cols + time_sig_cols

missing = [c for c in feature_cols if c not in songs_gmm.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

songs_gmm[feature_cols] = songs_gmm[feature_cols].astype(float)

# Primary artist (first listed)
songs_gmm['primary_artist'] = (
    songs_gmm['track_artist'].fillna('').str.split(',').str[0].str.strip()
)

# --- Fit GMM ---
n_components = 8
X = songs_gmm[feature_cols].to_numpy()

gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
gmm.fit(X)

# Hard cluster assignment (most likely component)
songs_gmm['gmm_cluster'] = gmm.predict(X)

# Soft cluster probabilities — shape (n_songs, n_components)
# These are the true GMM posteriors: P(component k | song)
cluster_probs = gmm.predict_proba(X)
prob_cols = [f'gmm_prob_{i}' for i in range(n_components)]
songs_gmm[prob_cols] = cluster_probs

print(f"GMM fitted: {n_components} components, {len(feature_cols)} features.")
print(f"Songs per cluster:\n{songs_gmm['gmm_cluster'].value_counts().sort_index().to_string()}")


GMM fitted: 8 components, 47 features.
Songs per cluster:
gmm_cluster
0     541
1    1019
2    1093
3     409
4     370
5     346
6     256
7     459


In [4]:
# ---------------------------------------------------------------
# GMM RECOMMENDATION FUNCTION
# ---------------------------------------------------------------
# "Small artist" = artist whose best-known track has popularity
# below this threshold (0–1 scale; 0.6 → top track scored < 60/100)
SMALL_ARTIST_THRESHOLD = 0.6


def recommend_small_artists_gmm(input_song, input_artist=None, n_recs=3):
    """
    Recommend songs from smaller/underground artists that are musically
    similar to a given popular input song.

    How it uses the GMM properly:
      Each song has a soft cluster distribution P(component k | song)
      computed from gmm.predict_proba().  Similarity between two songs
      is measured as the dot product of their cluster probability vectors:

          GMM_sim(seed, candidate) = Σ_k  P(k|seed) · P(k|candidate)

      This is a proper probabilistic measure: it is high when both songs
      have high probability mass on the same latent audio components.
      It is combined 50/50 with cosine similarity in raw feature space.

    Parameters
    ----------
    input_song   : str  — track name of the seed (popular) song
    input_artist : str  — (optional) artist name to disambiguate
    n_recs       : int  — number of recommendations to return (default 3)

    Returns
    -------
    seed : pd.Series       — the matched seed song row
    recs : pd.DataFrame    — top-n recommended songs
    """

    # 1. Find the seed song ----------------------------------------
    mask = songs_gmm['track_name'].str.lower() == input_song.lower()
    matches = songs_gmm[mask].copy()

    if input_artist:
        am = matches['track_artist'].str.lower().str.contains(
            input_artist.lower(), na=False
        )
        if am.any():
            matches = matches[am]

    # Fallback: partial name match
    if matches.empty:
        mask = songs_gmm['track_name'].str.lower().str.contains(
            input_song.lower(), na=False
        )
        matches = songs_gmm[mask].copy()

    if matches.empty:
        raise ValueError(f"Song '{input_song}' not found in dataset.")

    # Use the most popular match as the seed
    seed = matches.sort_values('track_popularity', ascending=False).iloc[0]

    seed_vec   = seed[feature_cols].to_numpy(dtype=float).reshape(1, -1)
    seed_probs = seed[prob_cols].to_numpy(dtype=float)   # (n_components,)

    # 2. Candidate pool: small artists only, exclude seed ----------
    candidates = songs_gmm[
        (songs_gmm['artist_top_track_popularity'] < SMALL_ARTIST_THRESHOLD) &
        (songs_gmm['track_id'] != seed['track_id'])
    ].copy()

    if candidates.empty:
        raise ValueError(
            f"No small-artist songs found (threshold={SMALL_ARTIST_THRESHOLD})."
        )

    # 3. Score candidates ------------------------------------------
    # (a) Proper GMM similarity: dot product of posterior distributions
    cand_probs  = candidates[prob_cols].to_numpy(dtype=float)   # (n, n_components)
    gmm_sim     = cand_probs @ seed_probs                        # (n,)

    # (b) Cosine similarity in audio feature space
    cand_feats  = candidates[feature_cols].to_numpy(dtype=float)
    cos_sim     = cosine_similarity(seed_vec, cand_feats).flatten()

    # (c) Combined score — equal weight
    candidates['gmm_similarity'] = gmm_sim
    candidates['cos_similarity'] = cos_sim
    candidates['score']          = 0.5 * gmm_sim + 0.5 * cos_sim

    # 4. Return top n_recs -----------------------------------------
    recs = (
        candidates
        .sort_values('score', ascending=False)
        .head(n_recs)
        [['track_name', 'track_artist', 'track_popularity',
          'artist_top_track_popularity', 'gmm_cluster',
          'gmm_similarity', 'cos_similarity', 'score']]
        .reset_index(drop=True)
    )

    return seed, recs


# ---------------------------------------------------------------
# DEMO  — change input_song to any track in the dataset
# ---------------------------------------------------------------
input_song   = "Don't Cry (Original)"
input_artist = None          # set e.g. "Tate McRae" to disambiguate

seed, recs = recommend_small_artists_gmm(
    input_song, input_artist=input_artist, n_recs=3
)

print("=" * 62)
print("INPUT SONG")
print(f"  Title  : {seed['track_name']}")
print(f"  Artist : {seed['track_artist']}")
print(f"  Track popularity          : {seed['track_popularity']:.2f}")
print(f"  Artist top-track pop.     : {seed['artist_top_track_popularity']:.2f}")
print(f"  GMM cluster               : {seed['gmm_cluster']}")
print("=" * 62)
print(f"\nTOP {len(recs)} RECOMMENDATIONS FROM SMALLER ARTISTS\n")

for i, row in recs.iterrows():
    print(f"  {i+1}. \"{row['track_name']}\"  by  {row['track_artist']}")
    print(f"     Artist top-track popularity : {row['artist_top_track_popularity']:.2f}")
    print(f"     GMM sim : {row['gmm_similarity']:.4f}  |  "
          f"Cosine sim : {row['cos_similarity']:.4f}  |  "
          f"Score : {row['score']:.4f}")
    print()


INPUT SONG
  Title  : Don't Cry (Original)
  Artist : Guns N' Roses
  Track popularity          : 0.72
  Artist top-track pop.     : 0.84
  GMM cluster               : 2

TOP 3 RECOMMENDATIONS FROM SMALLER ARTISTS

  1. "My Girl"  by  Mayflower
     Artist top-track popularity : 0.28
     GMM sim : 1.0000  |  Cosine sim : 0.9958  |  Score : 0.9979

  2. "Here Without You"  by  3 Doors Down
     Artist top-track popularity : 0.50
     GMM sim : 1.0000  |  Cosine sim : 0.9956  |  Score : 0.9978

  3. "Sad In Carolina"  by  Dexter and The Moonrocks
     Artist top-track popularity : 0.17
     GMM sim : 1.0000  |  Cosine sim : 0.9913  |  Score : 0.9957

